# Real-time Groundwater Resource Evaluation using Image Processing and IoT
## Member 2: Image Models & Explainable AI (XAI) Suite
### Complete Pipeline: Zenodo Dataset + CLAHE Preprocessing + 5 Algorithms Benchmark
**Dataset DOI**: [10.5281/zenodo.14537782](https://doi.org/10.5281/zenodo.14537782) (Nagkoulis et al., 2025)

---
### Pipeline Overview:
1. **Load Dataset**: Download and extract the Sentinel-2 / Groundwater dataset from Zenodo.
2. **Data Preprocessing & CLAHE Algorithm**: Apply Contrast Limited Adaptive Histogram Equalization (CLAHE) in LAB color space to enhance local surface moisture and vegetation contrast.
3. **Train All 5 Algorithms** with Classification Heads:
   - **Algorithm 1: Swin Transformer** (Shifted Windows Vision Transformer)
   - **Algorithm 2: SegFormer** (Hierarchical Mix Transformer - MiT-B0)
   - **Algorithm 3: Attention U-Net** (Spatial Attention Gate CNN)
   - **Algorithm 4: DeepLabV3+** (Atrous Spatial Pyramid Pooling - ASPP)
   - **Algorithm 5: ResNet-50** (Residual Convolutional Baseline)
4. **Performance Leaderboard & Evaluation**: Compare Test Accuracy, Macro F1, and Loss.
5. **Explainable AI (XAI)**: Generate Grad-CAM heatmaps across all 5 architectures.

### Step 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision transformers grad-cam lime shap opencv-python netCDF4 xarray scikit-learn matplotlib tqdm

### Step 2: Download & Extract Zenodo Dataset (~3.4 GB)
*High speed direct download (~100 MB/s on Google Colab/Cloud)*

In [ ]:
import os
import urllib.request

zenodo_url = "https://zenodo.org/records/14537783/files/initial_data.rar?download=1"
rar_file = "initial_data.rar"

if not os.path.exists(rar_file):
    print("Downloading initial_data.rar from Zenodo...")
    !wget -O initial_data.rar "https://zenodo.org/records/14537783/files/initial_data.rar?download=1"
else:
    print("initial_data.rar already downloaded.")

# Extract RAR file
!mkdir -p zenodo_raw
!unrar x -o+ initial_data.rar zenodo_raw/

### Step 3: Perform CLAHE (Contrast Limited Adaptive Histogram Equalization) on All Images

**Why CLAHE for Groundwater Imagery?**
- Standard histogram equalization over-amplifies noise in flat areas (e.g. desert or deep water).
- CLAHE computes histograms over local tiles ($8 \times 8$) and clips contrast beyond a threshold (clip limit 2.0).
- Applying CLAHE to the **Luminance ($L$) channel in LAB color space** enhances subtle moisture boundaries and crop stress without distorting natural spectral colors.

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def apply_clahe(img_pil, clip_limit=2.0, tile_grid_size=(8, 8)):
    img_np = np.array(img_pil.convert('RGB'))
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_enhanced = clahe.apply(l)
    lab_enhanced = cv2.merge((l_enhanced, a, b))
    rgb_enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
    return Image.fromarray(rgb_enhanced)

# Batch Preprocess Dataset splits to 'data_clahe/'
def batch_clahe_preprocess(input_root="data", output_root="data_clahe"):
    count = 0
    for split in ["train", "val", "test"]:
        for cls in ["0_severe_depletion", "1_moderate_stress", "2_safe_abundant"]:
            in_dir = os.path.join(input_root, split, cls)
            out_dir = os.path.join(output_root, split, cls)
            if not os.path.exists(in_dir): continue
            os.makedirs(out_dir, exist_ok=True)
            for f in os.listdir(in_dir):
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                    im = Image.open(os.path.join(in_dir, f))
                    im_clahe = apply_clahe(im)
                    im_clahe.save(os.path.join(out_dir, f))
                    count += 1
    print(f"Successfully preprocessed {count} images with CLAHE into '{output_root}/'!")

batch_clahe_preprocess()

### Step 4: DataLoaders with PyTorch Transformations

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

img_size = (224, 224)
train_tf = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_dir = "data_clahe" if os.path.exists("data_clahe") else "data"
train_loader = DataLoader(datasets.ImageFolder(f"{data_dir}/train", transform=train_tf), batch_size=16, shuffle=True)
val_loader = DataLoader(datasets.ImageFolder(f"{data_dir}/val", transform=eval_tf), batch_size=16, shuffle=False)
test_loader = DataLoader(datasets.ImageFolder(f"{data_dir}/test", transform=eval_tf), batch_size=16, shuffle=False)
classes = train_loader.dataset.classes
print(f"Loaded {len(classes)} Classes from {data_dir}: {classes}")

### Step 5: Define All 5 Architectures (with Classification Heads)

1. **Swin Transformer**: Hierarchical shifted local window self-attention.
2. **SegFormer (MiT-B0)**: Positional-free multi-scale Mix Transformer.
3. **Attention U-Net**: Attention Gates highlight salient moisture/crop features.
4. **DeepLabV3+**: Atrous Spatial Pyramid Pooling (ASPP) captures multi-scale features.
5. **ResNet-50**: Standard deep residual convolutional baseline.

In [ ]:
import torch.nn as nn
from torchvision.models import swin_t, Swin_T_Weights, resnet50, ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from transformers import SegformerForImageClassification

# 1. Swin Transformer
class SwinClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = swin_t(weights=Swin_T_Weights.DEFAULT)
        in_f = self.model.head.in_features
        self.model.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x): return self.model(x)

# 2. SegFormer
class SegFormerModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = SegformerForImageClassification.from_pretrained("nvidia/mit-b0", num_labels=num_classes, ignore_mismatched_sizes=True)
    def forward(self, x): return self.model(pixel_values=x).logits

# 3. Attention U-Net Classifier
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True),
                                  nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True))
    def forward(self, x): return self.conv(x)

class AttentionUNetClassifier(nn.Module):
    def __init__(self, in_c=3, num_classes=3, base_f=32):
        super().__init__()
        self.c1 = ConvBlock(in_c, base_f); self.p1 = nn.MaxPool2d(2)
        self.c2 = ConvBlock(base_f, base_f*2); self.p2 = nn.MaxPool2d(2)
        self.c3 = ConvBlock(base_f*2, base_f*4); self.p3 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_f*4, base_f*8)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(base_f*8, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, num_classes))
    def forward(self, x):
        b = self.bottleneck(self.p3(self.c3(self.p2(self.c2(self.p1(self.c1(x)))))))
        return self.classifier(self.gap(b))

# 4. DeepLabV3+ Classifier
class DeepLabV3Classifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        base = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
        self.backbone = base.backbone
        self.aspp = base.classifier[0] # ASPP module
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, num_classes))
    def forward(self, x): return self.classifier(self.gap(self.aspp(self.backbone(x)['out'])))

# 5. ResNet-50 Classifier
class ResNetClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)
        in_f = self.model.fc.in_features
        self.model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x): return self.model(x)

### Step 6: Train All 5 Algorithms & Benchmark

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, accuracy_score

models_dict = {
    "Swin Transformer": (SwinClassifier(len(classes)), 1e-4),
    "SegFormer (MiT-B0)": (SegFormerModel(len(classes)), 5e-5),
    "Attention U-Net": (AttentionUNetClassifier(num_classes=len(classes)), 3e-4),
    "DeepLabV3+": (DeepLabV3Classifier(num_classes=len(classes)), 1e-4),
    "ResNet-50": (ResNetClassifier(len(classes)), 3e-4)
}

leaderboard = []
criterion = nn.CrossEntropyLoss()
epochs = 10

for name, (model, lr) in models_dict.items():
    print(f"\n>>> Training {name} on CLAHE Dataset (LR: {lr})...")
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    best_f1 = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        preds, targets = [], []
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            targets.extend(lbls.cpu().numpy())
        scheduler.step()

    # Test evaluation
    model.eval()
    test_preds, test_targets = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            test_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            test_targets.extend(lbls.numpy())

    test_acc = accuracy_score(test_targets, test_preds)
    test_f1 = f1_score(test_targets, test_preds, average='macro')
    print(f"Result for {name}: Test Acc = {test_acc*100:.2f}% | Macro F1 = {test_f1*100:.2f}%")
    leaderboard.append({"name": name, "acc": test_acc, "f1": test_f1})

# Display Leaderboard
leaderboard = sorted(leaderboard, key=lambda x: x['f1'], reverse=True)
print("\n" + "="*50)
print(f"{'Rank':<6}{'Algorithm':<22}{'Test Accuracy':<15}{'Macro F1'}")
print("="*50)
for r, item in enumerate(leaderboard, 1):
    print(f"{r:<6}{item['name']:<22}{item['acc']*100:<14.2f}%{item['f1']*100:.2f}%")
print("="*50)

### Step 7: Benchmark Bar Chart & Explainable AI (Grad-CAM)

In [ ]:
# Plot Benchmark Comparison Bar Chart
names = [x['name'] for x in leaderboard]
accs = [x['acc']*100 for x in leaderboard]
f1s = [x['f1']*100 for x in leaderboard]

plt.figure(figsize=(10, 5))
x_pos = range(len(names))
plt.bar([i-0.2 for i in x_pos], accs, width=0.4, label='Accuracy (%)', color='#3b82f6')
plt.bar([i+0.2 for i in x_pos], f1s, width=0.4, label='Macro F1 (%)', color='#10b981')
plt.xticks(x_pos, names, rotation=15)
plt.ylabel('Score (%)')
plt.title('5-Algorithm Groundwater Benchmark on CLAHE Preprocessed Data')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()